# 11 · Profile: O*NET 31.0, what each job takes (skills, knowledge, tools, experience)

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/onet.md`.

**Why this source matters.** brief question 2 asks *what does each job take to get in?* The BLS tables (nb 04) give one line per job: typical degree, experience, training. **O*NET (the U.S. Department of Labor's Occupational Information Network) goes deeper: for about 900 jobs it rates 35 skills, 33 knowledge areas, 41 work activities, lists the software and technology used on the job, and asks workers how much education and prior experience the job really takes. It is also the base that every published AI-exposure score** (brief S12) is built on, so it decides how that score will join.

The brief uses it three ways:
1. **Skills per job** (Q2): the top skills and tools a student would need.
2. **Junior vs senior** (O1): O*NET's "related work experience" answers are a candidate for one of the three estimates.
3. The join key for the AI score (S12) and a second opinion on BLS entry education.

**What this notebook does:** reads the 31.0 database (August 2026), checks grain, codes, and what changed from 30.0, measures how much of NYC's employment it covers, and compares it with BLS. It changes nothing.

In [1]:
import io
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NY_MSA = "35620"
CODE = "O*NET-SOC Code"

man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key.eq("onet") | man.file.str.endswith(("oesm25ma.zip", "education_2026-10-12.xlsx", "CIP2020_SOC2018_Crosswalk.xlsx"))]
display(FILES[["key", "file", "downloaded_utc", "bytes", "note"]])

,key,file,downloaded_utc,bytes,note
7,bls_oews_metro,jobs/bls_oews_metro/oesm25ma.zip,2026-10-12T17:03:33+00:00,39932338,"OEWS May 2025, all metro areas (NY = 35620)"
16,bls_ep_education,jobs/bls_ep_education/education_2026-10-12.xlsx,2026-10-12T17:05:26+00:00,118332,"Entry education, experience, training; attainment table 5.3"
84,nces_cip,graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx,2026-10-12T17:08:46+00:00,428901,Official major -> job crosswalk
111,onet,skills/onet/db_31_0_csv.zip,2026-10-12T17:15:19+00:00,16237378,O*NET 31.0 (Aug 2026)
112,onet,skills/onet/db_30_0_text.zip,2026-10-12T17:15:31+00:00,13462590,O*NET 30.0: last with Tools Used
113,onet,skills/onet/Education_CIP_to_ONET_SOC.xlsx,2026-10-12T17:15:35+00:00,187053,
196,onet,skills/onet/2019_to_SOC_Crosswalk.xlsx,2026-10-13T03:49:45+00:00,44847,O*NET-SOC 2019 -> SOC 2018 (re-fetched 2026-10-12: first copy was ...


## 0. Source header

| | **O*NET 31.0 Database** (`onet`) |
|---|---|
| **Publisher** | National Center for O*NET Development, for the U.S. Department of Labor, Employment and Training Administration |
| **Files** | `db_31_0_csv.zip` (46 CSV tables + a readme, August 2026), `db_30_0_text.zip` (the previous release: kept because it is the last one with *Tools Used*, and to see what changed), `2019_to_SOC_Crosswalk.xlsx` (O*NET-SOC 2019 → SOC 2018), `Education_CIP_to_ONET_SOC.xlsx` (majors → O*NET jobs) |
| **Release** | 31.0, August 2026. O*NET updates about 100 jobs per release (each job is resurveyed every ~5–10 years) |
| **How it was collected** | **Survey + expert ratings.** Workers in each job (*Incumbent*) rate knowledge, activities, context, education and experience; *Occupational Experts* fill jobs that are hard to sample; trained *Analysts* rate skills and abilities from that evidence. Software lists come from job postings and employer data |
| **Who it covers** | **United States, not NYC.** One national profile per job. No wages, no counts |
| **Codes** | O*NET-SOC 2019 (8 characters, `15-1252.00`): the SOC 2018 code + a 2-digit suffix. `.00` is the SOC job itself; `.01`, `.02`… are O*NET-only specialties |
| **License** | Creative Commons Attribution 4.0 (credit O*NET and the U.S. DOL/ETA) |
| **Documentation** | [O*NET 31.0 data dictionary](https://www.onetcenter.org/dictionary/31.0/csv/), [O*NET-SOC taxonomy](https://www.onetcenter.org/taxonomy.html) |

## 1. Load as text

Read every table in the zip as text, then list what's there.

In [2]:
P = {Path(f).name: RAW / f for f in FILES.file}
z31 = zipfile.ZipFile(P["db_31_0_csv.zip"])
T = {Path(n).stem: pd.read_csv(z31.open(n), dtype=str, keep_default_na=False)
     for n in z31.namelist() if n.endswith(".csv")}
print((z31.read("db_31_0_csv/Read Me.txt").decode()[:120]).strip())
inv = pd.DataFrame([{"table": k, "rows": len(d), "cols": d.shape[1],
                     "jobs": d[CODE].nunique() if CODE in d else None} for k, d in T.items()]).sort_values("table")
print(f"{len(T)} tables, {inv.rows.sum():,} rows")
display(inv.set_index("table").T)

O*NET 31.0 Database
August 2026 Release
-------------------

For information on current and future releases of the O*NET
45 tables, 1,120,006 rows


table,abilities,abilities_to_work_activities,abilities_to_work_context,career_interest_type_keywords,career_interest_types,content_model_reference,education,education_categories,emerging_tasks,essential_skills,essential_skills_to_work_activities,essential_skills_to_work_context,gwas_to_iwas,gwas_to_iwas_to_dwas,interests_illustrative_activities,interests_illustrative_occupations,job_titles,job_zone_reference,job_zones,knowledge,...,scales_reference,software_skills,specific_interest_areas,specific_interest_areas_to_career_interest_types,survey_booklet_locations,task_categories,task_ratings,task_statements,tasks_to_dwas,training_and_experience,training_and_experience_categories,transferable_skills,transferable_skills_to_work_activities,transferable_skills_to_work_context,work_activities,work_context,work_context_categories,work_styles,work_styles_to_work_activities,work_styles_to_work_context
rows,94640.0,381.0,139.0,75.0,8307.0,3006.0,11495.0,24.0,338.0,18200.0,110.0,39.0,332.0,2087.0,188.0,186.0,54269.0,4.0,923.0,60060.0,...,33.0,31821.0,73062.0,53.0,211.0,7.0,165780.0,18838.0,24087.0,26812.0,29.0,45500.0,122.0,57.0,74702.0,305389.0,281.0,37422.0,303.0,266.0
cols,15.0,4.0,4.0,4.0,9.0,3.0,15.0,6.0,8.0,15.0,4.0,4.0,4.0,6.0,4.0,5.0,5.0,7.0,5.0,15.0,...,4.0,7.0,9.0,4.0,5.0,4.0,15.0,8.0,8.0,15.0,6.0,15.0,4.0,4.0,15.0,16.0,6.0,9.0,4.0,4.0
jobs,910.0,NaN,NaN,NaN,923.0,NaN,901.0,NaN,245.0,910.0,NaN,NaN,NaN,NaN,NaN,164.0,1016.0,NaN,923.0,910.0,...,NaN,923.0,891.0,NaN,NaN,NaN,911.0,923.0,923.0,901.0,NaN,910.0,NaN,NaN,911.0,911.0,NaN,891.0,NaN,NaN


The tables come in three kinds:
- **Ratings per job** (one number per job × descriptor × scale): skills, knowledge, abilities, work activities, work context, work styles.
- **Category answers per job** (share of workers giving each answer): education, training and experience, task frequency.
- **Lists and references**: occupation titles, job zones, task statements, software, code lists for the scales and categories.

v1 needs the first kind for skills and knowledge, the second for education and experience, plus job zones and the software list.

**Load findings**
- **45 tables, 1.12M rows**, all read cleanly as text. Biggest: work context (305k), task ratings (166k), abilities (95k), work activities (75k).
- **1,016 O*NET jobs in the job list; ratings exist for 910; job zones, tasks and software for 923**.

## 2. What one row is (grain)

**Presumed key** for a ratings table: job + descriptor (`Element ID`) + scale (`Scale ID`). Category tables add `Category`; the task table adds `Task ID`; the software table is job + product.

In [3]:
KEYS = {
    "essential_skills": [CODE, "Element ID", "Scale ID"],
    "transferable_skills": [CODE, "Element ID", "Scale ID"],
    "knowledge": [CODE, "Element ID", "Scale ID"],
    "abilities": [CODE, "Element ID", "Scale ID"],
    "work_activities": [CODE, "Element ID", "Scale ID"],
    "work_styles": [CODE, "Element ID", "Scale ID"],
    "work_context": [CODE, "Element ID", "Scale ID", "Category"],
    "education": [CODE, "Element ID", "Scale ID", "Category"],
    "training_and_experience": [CODE, "Element ID", "Scale ID", "Category"],
    "task_ratings": [CODE, "Task ID", "Scale ID", "Category"],
    "task_statements": [CODE, "Task ID"],
    "job_zones": [CODE],
    "occupation_data": [CODE],
    "software_skills": [CODE, "Workplace Example", "Element ID"],
}
display(pd.DataFrame([{"table": t, "key": " + ".join(k), "rows": len(T[t]), "duplicates": int(T[t].duplicated(k).sum())}
                      for t, k in KEYS.items()]).set_index("table"))

,key,rows,duplicates
table,,,
essential_skills,O*NET-SOC Code + Element ID + Scale ID,18200,0
transferable_skills,O*NET-SOC Code + Element ID + Scale ID,45500,0
knowledge,O*NET-SOC Code + Element ID + Scale ID,60060,0
abilities,O*NET-SOC Code + Element ID + Scale ID,94640,0
work_activities,O*NET-SOC Code + Element ID + Scale ID,74702,0
work_styles,O*NET-SOC Code + Element ID + Scale ID,37422,0
work_context,O*NET-SOC Code + Element ID + Scale ID + Category,305389,0
education,O*NET-SOC Code + Element ID + Scale ID + Category,11495,0
training_and_experience,O*NET-SOC Code + Element ID + Scale ID + Category,26812,0


**Grain findings**
- Every presumed key is unique (14 tables, 0 duplicates). One row of a ratings table = one job × one descriptor × one scale (`IM` importance 1–5, `LV` level 0–7).
- Category tables (education, experience, work context, task frequency) have one row per answer category: the value is the *share of workers* giving that answer, so they sum to ~100 per job and question.

## 3. Column profile

### 3a. Who rated it, when, and how sure

In [4]:
RATED = ["essential_skills", "transferable_skills", "knowledge", "abilities", "work_activities", "education", "training_and_experience"]
rows = []
for t in RATED:
    d = T[t]
    n = pd.to_numeric(d["N"], errors="coerce")
    yr = d.Date.str[-4:].astype(int)
    rows.append({"table": t, "jobs": d[CODE].nunique(), "elements": d["Element ID"].nunique(),
                 "scales": ",".join(sorted(d["Scale ID"].unique())),
                 **{f"src: {k}": f"{v:.0%}" for k, v in d["Domain Source"].value_counts(normalize=True).items()},
                 "median N": n.median(), "suppress=Y": f"{(d['Recommend Suppress'] == 'Y').mean():.1%}",
                 "rated 2022+": f"{(yr >= 2022).mean():.0%}", "oldest": yr.min()})
display(pd.DataFrame(rows).set_index("table").fillna(""))

,jobs,elements,scales,src: Analyst,median N,suppress=Y,rated 2022+,oldest,src: Incumbent,src: Occupational Expert,src: Analyst - Transition
table,,,,,,,,,,,
essential_skills,910,10,"IM,LV",100%,8.0,0.1%,52%,2010,,,
transferable_skills,910,25,"IM,LV",100%,8.0,0.3%,52%,2010,,,
knowledge,910,33,"IM,LV",,22.0,8.2%,52%,2005,72%,27%,1%
abilities,910,52,"IM,LV",100%,8.0,0.1%,52%,2004,,,
work_activities,911,41,"IM,LV",,22.0,1.4%,52%,2004,72%,27%,1%
education,901,2,"IM,RL,RQ",,22.0,4.2%,52%,2005,73%,27%,
training_and_experience,901,4,"IM,OJ,PT,RW",,22.0,4.1%,53%,2005,72%,28%,


In [5]:
# The scales: what IM, LV, RL, RW ... mean and their ranges
display(T["scales_reference"].set_index("Scale ID").loc[["IM", "LV", "RL", "RW", "PT", "OJ", "DR", "WI"]])

,Scale Name,Minimum,Maximum
Scale ID,,,
IM,Importance,1,5
LV,Level,0,7
RL,Required Level Of Education (Categories 1-12),0,100
RW,Related Work Experience (Categories 1-11),0,100
PT,On-Site Or In-Plant Training (Categories 1-9),0,100
OJ,On-The-Job Training (Categories 1-9),0,100
DR,Distinctiveness Rank,0,10
WI,Work Styles Impact,-3,3


### 3b. What changed from 30.0: the *Skills* table is gone

31.0 has no `skills` table. It has `essential_skills` and `transferable_skills` instead. Check that together they are the old 35 skills, with the same values.

In [6]:
z30 = zipfile.ZipFile(P["db_30_0_text.zip"])
rd30 = lambda n: pd.read_csv(z30.open(f"db_30_0_text/{n}.txt"), sep="\t", dtype=str, keep_default_na=False)
s31 = pd.concat([T["essential_skills"].assign(part="essential"), T["transferable_skills"].assign(part="transferable")])
s30 = rd30("Skills")
k = [CODE, "Element ID", "Scale ID"]
m = s31.merge(s30, on=k, how="outer", suffixes=("_31", "_30"), indicator=True)
both = m[m._merge == "both"]
changed = both["Data Value_31"] != both["Data Value_30"]
print(f"30.0 Skills: {len(s30):,} rows, {s30['Element ID'].nunique()} skills, {s30[CODE].nunique()} jobs")
print(f"31.0 essential + transferable: {len(s31):,} rows, {s31['Element ID'].nunique()} skills, {s31[CODE].nunique()} jobs")
print("Same 35 skill IDs:", set(s30["Element ID"]) == set(s31["Element ID"]))
print("Row match:", m._merge.value_counts().to_dict())
print(f"Values changed: {changed.sum():,} rows in {both[changed][CODE].nunique()} jobs, all re-rated in "
      f"{sorted(both[changed].Date_31.str[-4:].unique())}; jobs new in 31.0: {m[m._merge == 'left_only'][CODE].nunique()}")
display(s31.drop_duplicates("Element ID").groupby("part")["Element Name"].agg(lambda s: ", ".join(s)).to_frame())

30.0 Skills: 62,580 rows, 35 skills, 894 jobs
31.0 essential + transferable: 63,700 rows, 35 skills, 910 jobs
Same 35 skill IDs: True
Row match: {'both': 62580, 'left_only': 1120, 'right_only': 0}
Values changed: 6,720 rows in 141 jobs, all re-rated in ['2026']; jobs new in 31.0: 16


,Element Name
part,
essential,"Reading Comprehension, Active Listening, Writing, Speaking, Mathem..."
transferable,"Social Perceptiveness, Coordination, Persuasion, Negotiation, Inst..."


In [7]:
# Other changes: tables in 30.0 that 31.0 dropped or renamed, and the new work-styles format
t30 = {Path(n).stem for n in z30.namelist() if n.endswith(".txt")}
norm = lambda s: s.lower().replace(",", "").replace(" ", "_")
print("30.0 tables with no same-named 31.0 table:", sorted(t for t in t30 if norm(t) not in T))
print("31.0 tables with no same-named 30.0 table:", sorted(t for t in T if t not in {norm(x) for x in t30}))
ws30 = rd30("Work Styles")
print("\nWork styles 30.0 scales:", ws30["Scale ID"].unique().tolist(), ws30["Element ID"].nunique(), "styles",
      "| 31.0 scales:", T["work_styles"]["Scale ID"].unique().tolist(), T["work_styles"]["Element ID"].nunique(), "styles")
tech30 = rd30("Technology Skills")
print(f"Technology skills 30.0: {len(tech30):,} rows | 31.0 'software_skills': {len(T['software_skills']):,} rows, "
      f"{T['software_skills'][CODE].nunique()} jobs, {T['software_skills']['Workplace Example'].nunique():,} distinct products")

30.0 tables with no same-named 31.0 table: ['Alternate Titles', 'Basic Interests to RIASEC', 'DWA Reference', 'Education, Training, and Experience', 'Education, Training, and Experience Categories', 'IWA Reference', 'Interests', 'RIASEC Keywords', 'Read Me', 'Skills', 'Skills to Work Activities', 'Skills to Work Context', 'Technology Skills', 'Tools Used', 'UNSPSC Reference', 'Work Values']
31.0 tables with no same-named 30.0 table: ['career_interest_type_keywords', 'career_interest_types', 'education', 'education_categories', 'essential_skills', 'essential_skills_to_work_activities', 'essential_skills_to_work_context', 'gwas_to_iwas', 'gwas_to_iwas_to_dwas', 'job_titles', 'software_skills', 'specific_interest_areas', 'specific_interest_areas_to_career_interest_types', 'training_and_experience', 'training_and_experience_categories', 'transferable_skills', 'transferable_skills_to_work_activities', 'transferable_skills_to_work_context', 'work_styles_to_work_activities', 'work_styles_to_w

**Profile findings**
- **Who rated it:** skills and abilities are rated by analysts (median N = 8 analysts). Knowledge, work activities, education and experience come 72–73% from workers in the job, 27–28% from occupational experts (median N = 22 respondents).
- **How old:** about half of all ratings are from 2022 or later; the oldest go back to 2004–2005. Each row keeps its own `Date`, so the age can be shown.
- **Suppression:** O*NET flags ratings it recommends hiding (`Recommend Suppress = Y`): 8.2% of knowledge ratings, 4% of education/experience, under 1.5% elsewhere. Staging keeps the flag; the dashboard hides flagged values.
- **The *Skills* table was split, not changed. 31.0's `essential_skills` (10: reading, listening, writing, speaking, math, science, critical thinking, active learning, learning strategies, monitoring) + `transferable_skills` (25) are exactly the 35 skills of 30.0, same element IDs. 62,580 rows match; values differ only for 141 jobs re-rated in 2026; 16 jobs are new** in 31.0.
- **Other 31.0 changes:** *Technology Skills* is now `software_skills` (31.8k rows, 8,753 products; each flagged *Hot Technology* and/or *In Demand*). Tools Used, Work Values, Interests and Alternate Titles are gone (30.0 is the last release with Tools). Work styles were rebuilt: 21 styles on new scales (`DR` distinctiveness rank, `WI` impact −3 to +3) instead of 16 styles rated by importance, so they can't be compared across releases.

## 4. Codes: O*NET-SOC → SOC 2018

The rest of the project speaks SOC 2018 (6 digits). O*NET says its codes roll up by dropping the suffix. Check that against the official crosswalk, then see which jobs actually have ratings.

In [8]:
cw = pd.read_excel(P["2019_to_SOC_Crosswalk.xlsx"], header=3, dtype=str)
cw.columns = ["onet", "onet_title", "soc", "soc_title"]
print(f"Crosswalk: {len(cw):,} O*NET jobs -> {cw.soc.nunique()} SOC jobs; "
      f"'drop the suffix' rule holds for {(cw.onet.str[:7] == cw.soc).mean():.0%}")
print("Same job list as occupation_data:", set(cw.onet) == set(T["occupation_data"][CODE]))

occ = T["occupation_data"].rename(columns={CODE: "onet"})
occ["soc"] = occ.onet.str[:7]
occ["suffix"] = np.where(occ.onet.str.endswith(".00"), ".00 (the SOC job)", "specialty (.01+)")
rated = set(s31[CODE])
occ["has_ratings"] = occ.onet.isin(rated)
display(pd.crosstab(occ.suffix, occ.has_ratings, margins=True))

per_soc = occ.groupby("soc").agg(n=("onet", "size"), rated=("has_ratings", "sum"),
                                 zero_rated=("onet", lambda s: (s.str.endswith(".00") & s.isin(rated)).any()))
per_soc["case"] = np.select([per_soc.zero_rated & (per_soc.rated == 1), per_soc.zero_rated, per_soc.rated > 0],
                            [".00 is the only rated profile", ".00 rated + rated specialties", "only specialties rated"],
                            "no ratings at all")
display(per_soc.case.value_counts().to_frame("SOC jobs"))
print("Unrated O*NET jobs, examples:", occ[~occ.has_ratings].Title.head(8).tolist())

Crosswalk: 1,016 O*NET jobs -> 867 SOC jobs; 'drop the suffix' rule holds for 100%
Same job list as occupation_data: True


has_ratings,False,True,All
suffix,,,
.00 (the SOC job),106,761,867
specialty (.01+),0,149,149
All,106,910,1016


,SOC jobs
case,
.00 is the only rated profile,709
no ratings at all,82
.00 rated + rated specialties,52
only specialties rated,24


Unrated O*NET jobs, examples: ['Legislators', 'Education Administrators, All Other', 'Personal Service Managers, All Other', 'Managers, All Other', 'Business Operations Specialists, All Other', 'Financial and Investment Analysts', 'Financial Risk Specialists', 'Financial Specialists, All Other']


In [9]:
# When a SOC job has several rated profiles: does it matter whether we use .00 or the average of all of them?
s31["soc"] = s31[CODE].str[:7]
s31["v"] = s31["Data Value"].astype(float)
im = s31[s31["Scale ID"] == "IM"]
multi = per_soc.index[per_soc.rated > 1]
x = im[im.soc.isin(multi)]
a = x[x[CODE].str.endswith(".00")].groupby(["soc", "Element ID"]).v.mean()
b = x.groupby(["soc", "Element ID"]).v.mean()
d = (a - b).dropna().abs()
print(f"{len(multi)} SOC jobs with more than one rated profile. Skill importance (1-5 scale), .00 vs average of all "
      f"profiles: mean gap {d.mean():.2f}, 95th percentile {d.quantile(.95):.2f}, max {d.max():.2f}")
top5 = lambda g: set(g.nlargest(5).index)
same_top = [len(top5(a.loc[s]) & top5(b.loc[s])) for s in a.index.get_level_values(0).unique()]
print(f"Top-5 skills shared between the two methods: median {np.median(same_top):.0f} of 5")

67 SOC jobs with more than one rated profile. Skill importance (1-5 scale), .00 vs average of all profiles: mean gap 0.16, 95th percentile 0.47, max 1.21


Top-5 skills shared between the two methods: median 4 of 5


**Code findings**
- **O*NET-SOC → SOC 2018 = drop the suffix, confirmed for all 1,016 codes** by the official crosswalk (after re-downloading it: the first copy in `data/raw` was the HTML web page, not the xlsx; fixed in `src/sources.py`).
- 867 SOC jobs. How they're rated:

| Case | SOC jobs | Rule for v1 |
|---|---|---|
| Only `.00` rated | 709 | use it |
| `.00` rated + rated specialties | 52 | use `.00` |
| Only specialties rated (e.g. `.01`, `.02`, no `.00` profile) | 24 | average the specialties |
| Nothing rated | 82 | no skill profile; show tasks, software, job zone if O*NET has them |

- **The roll-up choice barely matters.** For the 67 jobs with more than one rated profile, `.00` vs the average of all profiles differs by 0.16 on the 1–5 importance scale (95th percentile 0.47), and the top 5 skills overlap 4 of 5 (median).

## 5. Coverage: how much of NYC's employment does O*NET describe?

NYC jobs = OEWS May 2025, New York metro, detailed jobs (nb 03). A job is covered if it has skill ratings: directly, or (for BLS group codes like `31-1120`) through the O*NET jobs under it.

In [10]:
zo = zipfile.ZipFile(P["oesm25ma.zip"])
msa = next(n for n in zo.namelist() if Path(n).name.startswith("MSA_"))
o = pd.read_excel(io.BytesIO(zo.read(msa)), dtype=str, keep_default_na=False)
o.columns = [c.strip().upper() for c in o.columns]
o = o[(o.AREA == NY_MSA) & (o.O_GROUP == "detailed")].copy()
o["emp"] = pd.to_numeric(o.TOT_EMP.str.replace(",", ""), errors="coerce")
rated_soc = set(im.soc)
prefix = lambda c: c.rstrip("0")  # 31-1120 -> 31-112: every code under the group starts with it

def match(c):
    if c in rated_soc:
        return "1 exact SOC job rated"
    if c.endswith("0") and any(s.startswith(prefix(c)) for s in rated_soc):
        return "2 BLS group code, jobs under it rated"
    if c in set(occ.soc):
        return "3 in O*NET, not rated yet"
    return "4 not in O*NET"

o["coverage"] = o.OCC_CODE.map(match)
cov = o.groupby("coverage").agg(jobs=("emp", "size"), nyc_employment=("emp", "sum"))
cov["share"] = (cov.nyc_employment / cov.nyc_employment.sum()).map("{:.1%}".format)
print(f"NY metro, May 2025: {len(o)} detailed jobs, {o.emp.sum():,.0f} employed")
display(cov)
display(o[o.coverage.str[0].isin(["2", "3", "4"])].sort_values("emp", ascending=False)
        [["OCC_CODE", "OCC_TITLE", "emp", "coverage"]].head(14))

NY metro, May 2025: 769 detailed jobs, 9,437,990 employed


,jobs,nyc_employment,share
coverage,,,
1 exact SOC job rated,710,8340240.0,88.4%
"2 BLS group code, jobs under it rated",7,722690.0,7.7%
"3 in O*NET, not rated yet",47,155050.0,1.6%
4 not in O*NET,5,220010.0,2.3%


,OCC_CODE,OCC_TITLE,emp,coverage
94335,31-1120,Home Health and Personal Care Aides,644700.0,"2 BLS group code, jobs under it rated"
94221,25-9045,"Teaching Assistants, Except Postsecondary",126920.0,4 not in O*NET
94024,13-2051,Financial and Investment Analysts,53870.0,"3 in O*NET, not rated yet"
94714,53-1047,First-Line Supervisors of Transportation and Material Moving Worke...,35860.0,4 not in O*NET
94004,13-1020,Buyers and Purchasing Agents,29360.0,"2 BLS group code, jobs under it rated"
94622,51-2090,Miscellaneous Assemblers and Fabricators,26880.0,"2 BLS group code, jobs under it rated"
94136,21-1018,"Substance Abuse, Behavioral Disorder, and Mental Health Counselors",26200.0,4 not in O*NET
94204,25-2052,"Special Education Teachers, Kindergarten and Elementary School",22270.0,4 not in O*NET
94308,29-2010,Clinical Laboratory Technologists and Technicians,16220.0,"2 BLS group code, jobs under it rated"
94146,21-1099,"Community and Social Service Specialists, All Other",14030.0,"3 in O*NET, not rated yet"


**Coverage findings** (NY metro, OEWS May 2025, 9.44M jobs)

| How a NYC job gets an O*NET profile | Jobs | Share of employment |
|---|---|---|
| Exact SOC job rated | 710 | **88.4%** |
| BLS group code ending in 0, jobs under it rated (`31-1120` home health + personal care aides = 645k) | 7 | 7.7% |
| BLS aggregate not ending in 0 (`25-9045` teaching assistants, `53-1047`, `21-1018`, `25-2052`, `51-2028`): the parts are rated; reach them through nb 06's aggregate → parts bridge | 5 | 2.3% |
| In O*NET but not rated yet | 47 | 1.6% |

- So 98.4% of NYC employment can get a skill profile. The rest are new SOC 2018 jobs O*NET hasn't surveyed: financial and investment analysts (`13-2051`, 54k NYC jobs), financial risk specialists, EMTs, and "all other" catch-alls. These still have tasks, software and a job zone; only the ratings are missing.
- `13-2051` matters for CUNY (Baruch's finance graduates). Label it "skill ratings not yet published by O*NET" rather than borrowing another job's profile.

## 6. Relationships: does O*NET's major → job crosswalk add anything?

O*NET publishes its own CIP 2020 → O*NET-SOC table. Compare it with the NCES CIP → SOC crosswalk from nb 06, at SOC level.

In [11]:
a = pd.read_excel(P["Education_CIP_to_ONET_SOC.xlsx"], header=3, dtype=str)
a.columns = ["cip", "cip_title", "onet", "onet_title"]
A = set(zip(a.cip, a.onet.str[:7]))
b = pd.read_excel(P["CIP2020_SOC2018_Crosswalk.xlsx"], sheet_name="CIP-SOC", dtype=str)
B = set(zip(b.CIP2020Code, b.SOC2018Code))
print(f"O*NET: {len(a):,} CIP -> O*NET job rows = {len(A):,} CIP -> SOC pairs | NCES: {len(B):,} pairs")
print(f"In both: {len(A & B):,} | O*NET only: {sorted(A - B)} | NCES only: {len(B - A)}, "
      f"of which placeholder 99-9999: {sum(s == '99-9999' for _, s in B - A)}")
for c in ["24.0101", "42.0101", "11.0701"]:
    print(f"{c}: O*NET -> {sorted(s for x, s in A if x == c)}")

O*NET: 8,501 CIP -> O*NET job rows = 5,724 CIP -> SOC pairs | NCES: 6,097 pairs
In both: 5,723 | O*NET only: [('46.0000', '47-2061')] | NCES only: 374, of which placeholder 99-9999: 194
24.0101: O*NET -> ['25-1199']
42.0101: O*NET -> ['11-9199', '19-3032', '19-3033', '19-3039', '19-4061', '25-1066']
11.0701: O*NET -> ['11-3021', '15-1212', '15-1221', '15-1231', '15-1243', '15-1251', '15-1252', '15-1253', '15-1254', '15-1255', '15-1299', '15-2051', '25-1021']


**Relationship findings**
- **O*NET's CIP → job table is the NCES crosswalk.** 5,723 of its 5,724 CIP → SOC pairs are in the NCES file; the only extra is `46.0000 → 47-2061`. The 374 NCES-only pairs are 194 `99-9999` placeholders + jobs O*NET doesn't carry.
- So it adds nothing for D2: Liberal Arts (`24.0101`) still links only to postsecondary teachers (`25-1199`). Don't load it; the NCES crosswalk (nb 06) stays the major → job bridge.

## 7. Tie-out: does O*NET agree with BLS?

O*NET has no totals to reproduce, so the tie-out is against the other national source on the same question: BLS typical entry education and experience (2025 edition, nb 04). They measure different things: BLS = *typical education needed to enter* (an analyst's call), O*NET = *what workers say is required to do the job today* (a distribution). So exact agreement isn't expected; large gaps are what to look for.

### 7a. Education: BLS typical entry vs O*NET's most common answer

In [12]:
ed = T["education"][T["education"]["Scale ID"] == "RL"].copy()
ed["v"], ed["cat"] = ed["Data Value"].astype(float), ed.Category.astype(int)
ed = ed[ed[CODE].str.endswith(".00")]
ed["soc"] = ed[CODE].str[:7]
cats = T["education_categories"].query("`Scale ID` == 'RL'").set_index("Category")["Category Description"]
print("O*NET answer categories:", {int(k): v[:40] for k, v in cats.items()})

ONET_TO_BLS = {1: "No formal educational credential", 2: "High school diploma or equivalent", 3: "Postsecondary nondegree award",
               4: "Some college, no degree", 5: "Associate's degree", 6: "Bachelor's degree", 7: "Bachelor's degree",
               8: "Master's degree", 9: "Master's degree", 10: "Doctoral or professional degree",
               11: "Doctoral or professional degree", 12: "Doctoral or professional degree"}
EDU_ORDER = ["No formal educational credential", "High school diploma or equivalent", "Some college, no degree",
             "Postsecondary nondegree award", "Associate's degree", "Bachelor's degree", "Master's degree",
             "Doctoral or professional degree"]
mode = ed.loc[ed.groupby("soc").v.idxmax(), ["soc", "cat"]]
mode["onet_edu"] = mode.cat.map(ONET_TO_BLS)
mode = mode.merge(ed[ed.cat >= 6].groupby("soc").v.sum().rename("onet_bach_plus_%"), on="soc", how="left")
mode = mode.merge(ed[ed.cat == 5].groupby("soc").v.sum().rename("onet_assoc_%"), on="soc", how="left")

bls = pd.read_excel(P["education_2026-10-12.xlsx"], sheet_name="Table 5.4", header=1, dtype=str, keep_default_na=False)
bls = bls[bls.iloc[:, 1].str.fullmatch(r"\d\d-\d{4}")]
bls.columns = ["title", "soc", "bls_edu", "bls_exp", "bls_ojt", "ooh"]
cmp_ = bls.merge(mode, on="soc")
lvl = {e: i for i, e in enumerate(EDU_ORDER)}
gap = cmp_.onet_edu.map(lvl) - cmp_.bls_edu.map(lvl)
print(f"\n{len(cmp_)} jobs in both. Same level: {(gap == 0).mean():.0%} | O*NET higher: {(gap > 0).mean():.0%} | O*NET lower: {(gap < 0).mean():.0%}")
display(pd.crosstab(cmp_.bls_edu, cmp_.onet_edu).reindex(index=EDU_ORDER, columns=EDU_ORDER).fillna(0).astype(int)
        .rename(columns=lambda c: c[:14]))

O*NET answer categories: {1: 'Less than a High School Diploma', 2: 'High School Diploma or the equivalent (s', 3: 'Post-Secondary Certificate', 4: 'Some College Courses', 5: "Associate's Degree or other 2-year degre", 6: "Bachelor's Degree", 7: 'Post-Baccalaureate Certificate', 8: "Master's Degree", 9: "Post-Master's Certificate", 10: 'First Professional Degree - awarded for ', 11: 'Doctoral Degree', 12: 'Post-Doctoral Training'}

612 jobs in both. Same level: 68% | O*NET higher: 21% | O*NET lower: 10%


onet_edu,No formal educ,High school di,"Some college,",Postsecondary,Associate's de,Bachelor's deg,Master's degre,Doctoral or pr
bls_edu,,,,,,,,
No formal educational credential,28,52,1,1,0,0,0,0
High school diploma or equivalent,12,180,3,23,7,16,1,1
"Some college, no degree",0,2,0,0,0,2,0,0
Postsecondary nondegree award,0,13,3,13,1,2,0,0
Associate's degree,0,7,0,3,12,9,0,0
Bachelor's degree,0,4,1,2,6,103,5,2
Master's degree,0,0,0,0,0,3,24,5
Doctoral or professional degree,0,0,0,0,0,0,8,57


In [13]:
# The two levels CUNY awards: what share of workers say the job needs an associate's / a bachelor's or more?
for lev in ["Associate's degree", "Bachelor's degree"]:
    x = cmp_[cmp_.bls_edu == lev]
    print(f"BLS typical entry = {lev} ({len(x)} jobs): O*NET median share saying associate's {x['onet_assoc_%'].median():.0f}%, "
          f"bachelor's or more {x['onet_bach_plus_%'].median():.0f}%")
show = ["29-1141", "15-1252", "29-2061", "13-2011", "43-4051", "29-2034"]
print("Not in both lists:", [c for c in show if c not in set(cmp_.soc)])
display(cmp_.set_index("soc").reindex([c for c in show if c in set(cmp_.soc)])[["title", "bls_edu", "onet_edu", "onet_assoc_%", "onet_bach_plus_%"]].round(0))

BLS typical entry = Associate's degree (31 jobs): O*NET median share saying associate's 27%, bachelor's or more 22%
BLS typical entry = Bachelor's degree (123 jobs): O*NET median share saying associate's 5%, bachelor's or more 82%
Not in both lists: ['13-2011', '29-2034']


,title,bls_edu,onet_edu,onet_assoc_%,onet_bach_plus_%
soc,,,,,
29-1141,Registered nurses,Bachelor's degree,Bachelor's degree,19.0,57.0
15-1252,Software developers,Bachelor's degree,Bachelor's degree,5.0,90.0
29-2061,Licensed practical and licensed vocational nurses,Postsecondary nondegree award,"Some college, no degree",16.0,11.0
43-4051,Customer service representatives,High school diploma or equivalent,High school diploma or equivalent,14.0,14.0


### 7b. Experience: BLS work experience vs O*NET related work experience

In [14]:
te = T["training_and_experience"]
rw = te[(te["Scale ID"] == "RW") & te[CODE].str.endswith(".00")].copy()
rw["v"], rw["cat"] = rw["Data Value"].astype(float), rw.Category.astype(int)
rw["soc"] = rw[CODE].str[:7]
rwcats = T["training_and_experience_categories"].query("`Scale ID` == 'RW'").set_index("Category")["Category Description"]
print("O*NET related-experience answers:", {int(k): v for k, v in rwcats.items()})
newcomer = rw[rw.cat <= 2].groupby("soc").v.sum().rename("onet_newcomer_%")   # none, or up to 1 month
two_plus = rw[rw.cat >= 7].groupby("soc").v.sum().rename("onet_2yrs_plus_%")   # over 2 years
exp = bls.merge(newcomer, on="soc").merge(two_plus, on="soc")
print(f"{len(exp)} jobs in both. Median share of workers by BLS experience category:")
display(exp.groupby("bls_exp")[["onet_newcomer_%", "onet_2yrs_plus_%"]].median().round(1)
        .reindex(["None", "Less than 5 years", "5 years or more"]).assign(jobs=exp.bls_exp.value_counts()))
n = pd.to_numeric(rw.drop_duplicates(CODE).N, errors="coerce")
print(f"Respondents per job for this question: median {n.median():.0f}, 10th percentile {n.quantile(.1):.0f}")

O*NET related-experience answers: {1: 'None', 2: 'Up to and including 1 month', 3: 'Over 1 month, up to and including 3 months', 4: 'Over 3 months, up to and including 6 months', 5: 'Over 6 months, up to and including 1 year', 6: 'Over 1 year, up to and including 2 years', 7: 'Over 2 years, up to and including 4 years', 8: 'Over 4 years, up to and including 6 years', 9: 'Over 6 years, up to and including 8 years', 10: 'Over 8 years, up to and including 10 years', 11: 'Over 10 years'}
728 jobs in both. Median share of workers by BLS experience category:


,onet_newcomer_%,onet_2yrs_plus_%,jobs
bls_exp,,,
None,23.0,27.5,620
Less than 5 years,12.7,54.3,82
5 years or more,0.1,79.3,26


Respondents per job for this question: median 23, 10th percentile 16


**Tie-out findings**
- **Education: O*NET and BLS agree on 68% of 612 jobs.** When they differ, O*NET is usually higher (21% vs 10% lower): workers report more schooling than BLS's *entry* level, which fits the two definitions (what workers have vs what employers require to enter).
- For the two levels CUNY awards: in jobs BLS calls associate's-entry (31 jobs), only 27% of workers say the job needs an associate's and 22% say a bachelor's or more, so these are mixed-entry jobs. In bachelor's-entry jobs (123), 82% say a bachelor's or more, a clear signal.
- RNs: BLS says bachelor's; O*NET workers say 57% bachelor's+, 19% associate's. Same story as nb 04 (24.9% of RNs hold an associate's): the associate RN path is real and must show.
- **Experience lines up with BLS** (728 jobs): in jobs BLS says need no experience, a median 23% of workers needed ≤ 1 month of related experience; "less than 5 years" 13%; "5 years or more" 0.1%. Workers needing 2+ years: 28% → 54% → 79%. The direction is right at every step.
- Caveat: a median of 23 respondents per job answer the experience question, so job-level shares are noisy (±10 points or more). Use bands, not exact percentages.

## 8. Fitness for purpose

### 8a. Q2: what a job profile would look like for a student

In [15]:
def profile(soc):
    code_ = f"{soc}.00"
    sk = s31[(s31[CODE] == code_) & (s31["Scale ID"] == "IM")].nlargest(6, "v")
    kn = T["knowledge"][(T["knowledge"][CODE] == code_) & (T["knowledge"]["Scale ID"] == "IM")].assign(
        v=lambda d: d["Data Value"].astype(float)).nlargest(4, "v")
    sw = T["software_skills"][T["software_skills"][CODE] == code_]
    jz = T["job_zones"].set_index(CODE)["Job Zone"].get(code_)
    print(f"\n{T['occupation_data'].set_index(CODE).Title[code_]} ({soc}) | job zone {jz}")
    print("  top skills:   ", ", ".join(f"{n} {v:.1f}" for n, v in zip(sk["Element Name"], sk.v)))
    print("  top knowledge:", ", ".join(f"{n} {v:.1f}" for n, v in zip(kn["Element Name"], kn.v)))
    print(f"  software: {len(sw)} listed, {(sw['Hot Technology'] == 'Y').sum()} 'hot', {(sw['In Demand'] == 'Y').sum()} 'in demand':",
          ", ".join(sw[sw["In Demand"] == "Y"]["Workplace Example"].head(8)))
for s in ["29-1141", "15-1252", "13-2011", "29-2034"]:
    profile(s)


Registered Nurses (29-1141) | job zone 4
  top skills:    Social Perceptiveness 4.1, Active Listening 4.0, Speaking 4.0, Critical Thinking 4.0, Coordination 4.0, Service Orientation 4.0
  top knowledge: Psychology 4.6, Customer and Personal Service 4.4, Medicine and Dentistry 4.4, English Language 4.2
  software: 43 listed, 16 'hot', 0 'in demand': 

Software Developers (15-1252) | job zone 4
  top skills:    Programming 4.0, Critical Thinking 3.9, Judgment and Decision Making 3.6, Reading Comprehension 3.5, Active Learning 3.5, Systems Analysis 3.5
  top knowledge: Computers and Electronics 4.8, Mathematics 3.6, Customer and Personal Service 3.6, English Language 3.3
  software: 430 listed, 152 'hot', 30 'in demand': Amazon Web Services AWS software, Apache Kafka, Atlassian JIRA, C, C#, C++, Cascading style sheets CSS, Docker

Accountants and Auditors (13-2011) | job zone 4
  top skills:    Reading Comprehension 3.9, Active Listening 3.5, Writing 3.5, Critical Thinking 3.5, Complex P

In [16]:
# Does the skill profile separate associate-level from bachelor's-level jobs? (top skills by mean importance)
lev = bls.set_index("soc").bls_edu
imp = im[im[CODE].str.endswith(".00")].assign(level=lambda d: d.soc.map(lev))
t = imp[imp.level.isin(["Associate's degree", "Bachelor's degree"])].pivot_table(index="Element Name", columns="level", values="v")
t["gap"] = t["Bachelor's degree"] - t["Associate's degree"]
display(t.sort_values("gap").iloc[[0, 1, 2, -3, -2, -1]].round(2))

level,Associate's degree,Bachelor's degree,gap
Element Name,,,
Operation and Control,2.49,1.69,-0.80
Equipment Maintenance,2.02,1.22,-0.80
Repairing,1.99,1.21,-0.78
Writing,3.23,3.68,0.45
Negotiation,2.51,2.96,0.45
Operations Analysis,2.05,2.50,0.46


### 8b. O1 (junior vs senior) and S12 (AI score): how they would join

In [17]:
print(f"O1 candidate 'O*NET newcomer share' (workers who needed <= 1 month of related experience) exists for "
      f"{len(newcomer)} SOC jobs; NYC employment covered: "
      f"{o[o.OCC_CODE.isin(newcomer.index)].emp.sum() / o.emp.sum():.0%} (exact codes only)")
ts = T["task_statements"]
print(f"Task statements (what AI-exposure scores are built from): {len(ts):,} tasks for {ts[CODE].nunique()} jobs, "
      f"types {ts['Task Type'].value_counts().to_dict()}")
print("Every published AI-exposure score is keyed on O*NET-SOC (8-digit) or SOC (6-digit) codes, so it will join "
      "through the same suffix-drop rule as the skills.")

O1 candidate 'O*NET newcomer share' (workers who needed <= 1 month of related experience) exists for 752 SOC jobs; NYC employment covered: 86% (exact codes only)
Task statements (what AI-exposure scores are built from): 18,838 tasks for 923 jobs, types {'Core': 14071, 'Supplemental': 4349, '': 418}
Every published AI-exposure score is keyed on O*NET-SOC (8-digit) or SOC (6-digit) codes, so it will join through the same suffix-drop rule as the skills.


**Fitness findings**
- Q2 (what a job takes): strong. Each job gets top skills, top knowledge areas, job zone (1–5 preparation level), and a software list with *hot* / *in-demand* flags. Examples: software developers → Programming 4.0, Critical Thinking 3.9; 30 in-demand products (AWS, Kafka, Docker, Git…). Accountants → Reading Comprehension, Economics and Accounting 4.4; QuickBooks, Excel, SAP.
- Skills separate associate-level from bachelor's-level jobs the way you'd expect: associate jobs rate higher on equipment operation, maintenance and repair (+0.8); bachelor's jobs on writing, negotiation, operations analysis (+0.45).
- O1 (junior vs senior): usable as one of the three estimates. "Newcomer share" (workers who needed ≤ 1 month of related experience) exists for 752 SOC jobs, 86% of NYC employment by exact code, and moves with BLS's experience categories. Noisy per job, so show it as a band.
- S12 (AI score): the join is solved. The candidate scores are all keyed on O*NET-SOC or SOC codes, so they join through the same suffix-drop rule. O*NET's 18,838 task statements are what those scores rate.
- **Limits:** national, not NYC; one profile per job (no junior/senior split inside a job); no counts or wages; ratings 1–20 years old (show the date); some skill ratings for 2021+ jobs aren't published yet.

## 9. Issues list and proposed staging tables

### 9. Issues list

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | O*NET-SOC (8 chars) vs SOC 2018 (6) | all | `soc_code = left(onet_soc_code, 7)`; keep both columns |
| 2 | Several profiles per SOC job | 76 SOC jobs | Use `.00` when rated; else average the rated specialties. Flag `rollup_method` |
| 3 | BLS group/aggregate codes in OEWS (`31-1120`, `25-9045` …) | 12 NYC codes, 10% of employment | Profile = average of the rated parts, via nb 06's aggregate → parts bridge (built in core, not staging) |
| 4 | Jobs not rated yet (`13-2051`, `13-2054`, `29-2042` …) | 47 NYC codes, 1.6% of employment | No skill profile; show tasks, software, job zone; label "not yet rated by O*NET" |
| 5 | `Recommend Suppress = Y` | 8.2% of knowledge, ≤ 4% elsewhere | Keep as boolean; dashboard hides those values |
| 6 | `Not Relevant = Y` (level rating for an unimportant skill) | some `LV` rows | Keep as boolean; show `IM` (importance) by default, `LV` only when relevant |
| 7 | Ratings 2004–2026 | all | Keep `rated_date`; show "rated in YYYY" on the job page |
| 8 | Skills split into two tables in 31.0 | 63,700 rows | Stack both into one table with `skill_type` = essential / transferable |
| 9 | Work styles rebuilt in 31.0 (new scales) | 37k rows | Not in v1 (can't compare with older releases, not needed for Q2) |
| 10 | Experience/education answers from ~23 people per job | RW, RL | Store the shares; dashboard shows bands; MOE note |
| 11 | O*NET CIP → job table = NCES crosswalk | 8,501 rows | Don't load; NCES crosswalk stays the major → job bridge |
| 12 | Tools Used only in 30.0 | — | Not in v1 (software list covers what students need) |

### Proposed staging tables

All five keep `onet_soc_code`, `soc_code` (first 7 characters) and `onet_release = '31.0'`.

- **`stg_onet__occupation`**: one row per O*NET job (1,016): `onet_soc_code`, `soc_code`, `title`, `description`, `has_ratings`.
- **`stg_onet__descriptor_rating`**: one row per job × descriptor × scale, for skills (essential + transferable) and knowledge (123,760 rows, 910 jobs). Columns: `domain` (`skill_essential` / `skill_transferable` / `knowledge`), `element_id`, `element_name`, `scale` (`IM`/`LV`), `value`, `n`, `std_error`, `ci_low`, `ci_high`, `suppress`, `not_relevant`, `rated_date`, `rating_source`. Work activities and abilities can be added later with the same shape.
- **`stg_onet__education_experience`**: one row per job × question × answer category: required education (`RL`) and related work experience (`RW`) (19,007 rows, 901 jobs). Columns: `question`, `category`, `category_label`, `share_pct`, `n`, `suppress`, `rated_date`.
- **`stg_onet__job_zone`**: one row per job (923): `job_zone` 1–5, `rated_date`.
- **`stg_onet__software`**: one row per job × product (31,821): `product`, `commodity_id`, `commodity_title`, `is_hot_technology`, `is_in_demand`.

In [18]:
# Row accounting for the proposed staging tables (recomputed, not typed)
stg_skill = pd.concat([T[t].assign(domain=t)
                       for t in ["essential_skills", "transferable_skills", "knowledge"]])
print(f"stg_onet__descriptor_rating: skills + knowledge, IM and LV: {len(stg_skill):,} rows, "
      f"{stg_skill[CODE].nunique()} jobs; duplicates on key: {stg_skill.duplicated([CODE, 'Element ID', 'Scale ID']).sum()}")
cat = pd.concat([T["education"].query("`Scale ID` == 'RL'"), T["training_and_experience"].query("`Scale ID` == 'RW'")])
print(f"stg_onet__education_experience (RL + RW): {len(cat):,} rows, {cat[CODE].nunique()} jobs")
print(f"stg_onet__software: {len(T['software_skills']):,} rows | stg_onet__job_zone: {len(T['job_zones']):,} rows | "
      f"stg_onet__occupation: {len(T['occupation_data']):,} rows")

stg_onet__descriptor_rating: skills + knowledge, IM and LV: 123,760 rows, 910 jobs; duplicates on key: 0
stg_onet__education_experience (RL + RW): 19,007 rows, 901 jobs
stg_onet__software: 31,821 rows | stg_onet__job_zone: 923 rows | stg_onet__occupation: 1,016 rows
